# Five-Fold Cross-Validation for Vestibular Schwannoma Segmentation

This notebook runs 5-fold cross-validation for three 3D segmentation models on a
vestibular schwannoma (VS) MRI segmentation task, using the patch-based workflow in
fastMONAI, built on fastai, MONAI, and TorchIO.

## The task

Vestibular Schwannoma is a benign tumor arising from the vestibulocochlear nerve that can
cause hearing loss and brainstem compression ([Kujawa et al., 2024](https://doi.org/10.3389/fncom.2024.1365727)).
For small- to medium-sized lesions, guidelines recommend either upfront radiosurgery or
observation until radiographic tumor growth is evident ([Dhayalan et al., 2023](https://doi.org/10.1001/jama.2023.12222)).
Accurate, reproducible delineation of the tumor on Contrast-Enhanced T1-weighted (CE-T1w)
MRI supports treatment planning and volumetric follow-up. We treat it as a binary 3D
segmentation problem: for every voxel, decide tumor (1) or background (0).

## The dataset

We use 346 CE-T1w cases pooled from the Queen Square cohort
([Shapey et al., 2021](https://doi.org/10.1038/s41597-021-01064-w)) and the public crossMoDA
challenge data ([Dorent et al., 2023](https://doi.org/10.1016/j.media.2022.102628)). Each
case has a paired ground-truth tumor mask. The tumors are small relative to the field of
view, which is why we train on patches sampled around the lesion rather than on whole
volumes.

## What five-fold cross-validation buys us

Cross-validation gives a more honest estimate of how a model generalizes than a single
train/validation split. The 346 cases are partitioned into five folds. We train five times
per model; each run holds out one fold for validation and trains on the other four.
Averaging the held-out scores across the five folds uses every case for validation exactly
once and reports a mean and a spread rather than a single lucky (or unlucky) number.

The fold assignment is fixed ahead of time in the `fold` column of the dataset CSV
(values 1..5). Using a pre-assigned column, rather than reshuffling here, means every model
sees the exact same folds, so their scores are directly comparable, and the split is
reproducible across notebooks and machines. The CSV also carries a `split` column, but the
cross-validation intentionally ignores it: all 346 cases participate in CV.

## The three models

- **UNet** (MONAI built-in): the classic encoder-decoder with skip connections. A strong,
  well-understood convolutional baseline.
- **DynUNet** (MONAI built-in): a dynamic, nnU-Net-style UNet
  ([Isensee et al., 2024](https://doi.org/10.1007/978-3-031-72114-4_47)) with residual blocks
  and deep supervision (auxiliary losses at several decoder resolutions), which tends to
  train more stably on medical data.
- **SegMamba** (our fork): a state-space (Mamba) backbone for 3D segmentation. It replaces
  the quadratic-cost attention of a transformer with a linear-time selective scan, aiming to
  capture long-range 3D context efficiently.

All three are trained through an identical data, loss, metric, and evaluation pipeline, so
any difference in scores reflects the model rather than the plumbing.

Despite the high accuracy reported for automated VS segmentation, its real-world utility and
computational cost remain underexplored, and an apparent performance plateau raises the
question of whether newer architectures offer practical gains over a standard CNN
([Connor et al., 2025](https://doi.org/10.5152/iao.2025.241693); [Häußler et al., 2025](https://doi.org/10.1002/lary.31979)).
Comparing a standard convolutional baseline (UNet) against an nnU-Net-style model (DynUNet)
and a state-space model (SegMamba) on the same data is a controlled way to probe that
question. References cited throughout are listed at the end of this notebook.

## 1. Environment setup

We import the fastMONAI patch API, MONAI networks and losses, and the exact metric
functions used during evaluation. Everything runs in a conda environment with fastMONAI and
its dependencies installed. We `chdir` into the project folder early so that the relative
image and mask paths in the CSV (`../nii_data/...`) resolve correctly.

In [ ]:
import os
import gc
import json
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchio as tio

from monai.losses import DiceCELoss, DeepSupervisionLoss
from monai.networks.layers import Norm
from monai.networks.nets import UNet, DynUNet

from fastMONAI.vision_all import *

# SegMamba is an optional fork; UNet and DynUNet need none of it. Import it once here so a
# missing fork is reported now, with the install command, instead of mid-sweep. (A fork that
# imports but whose CUDA kernels are broken surfaces later, when the model is built, and is
# caught by the driver's try/except.)
_SEGMAMBA_INSTALL_HINT = (
    "  GPU (training):  pip install 'segmamba-v2[gpu] @ git+https://github.com/skaliy/SegMamba-V2.git'\n"
    "  CPU (inference): pip install 'segmamba-v2[cpu] @ git+https://github.com/skaliy/SegMamba-V2.git'"
)
try:
    from models_segmamba.segmambav2 import SegMamba
    SEGMAMBA_AVAILABLE = True
except ImportError as exc:
    SegMamba, SEGMAMBA_AVAILABLE = None, False
    print(f"[segmamba] fork not available ({exc}); UNet and DynUNet still run. To enable it:\n"
          + _SEGMAMBA_INSTALL_HINT)

# Resolve data paths against the repo root so they work wherever the kernel started
# (VS Code launches at the workspace root; plain Jupyter, in the notebook folder).
import fastMONAI
REPO_ROOT = Path(fastMONAI.__file__).resolve().parent.parent
os.chdir(REPO_ROOT / "research" / "vestibular_schwannoma")

## 2. Run configuration

The knobs below control the size of the run. A full paper run is
`3 models x 5 folds x 500 epochs`, which takes on the order of days on a single modern GPU.

To smoke-test the pipeline end to end in a few minutes instead, shrink the sweep, for
example:

- `MODELS_TO_RUN = ["unet"]`
- `FOLDS_TO_RUN  = [1]`
- `EPOCHS = 2`

The demo exercises every stage (preprocessing, patch loading, training, sliding-window
evaluation, metric computation, aggregation) without producing publication-quality models.
`TARGET_SPACING` and `PATCH_SIZE` are part of the shared preprocessing contract and must
match the inference notebook, so they are not casual knobs.

In [ ]:
# =============================== RUN KNOBS ===============================
# Which models and folds to run, and for how long. Shrink these for a quick
# pipeline smoke test (see the note above); keep the defaults for a full run.
MODELS_TO_RUN = ["unet", "dynunet", "segmamba"]
FOLDS_TO_RUN  = [1, 2, 3, 4, 5]

EPOCHS  = 500        # paper setting; set to e.g. 2 for a quick pipeline demo
BS      = 4          # patches per training batch
LR      = 1e-3       # maximum learning rate for fit_one_cycle
USE_TTA = True       # 8-flip test-time augmentation during evaluation
# ========================================================================

# Shared preprocessing contract. These MUST match 02_inference_new_cases.ipynb.
TARGET_SPACING = [0.4102, 0.4102, 1.5]   # resample target, voxel size in mm
PATCH_SIZE     = [192, 192, 48]          # patch shape in voxels

DATA_CSV = "ml_dataset.csv"

# cuDNN autotuner: a fixed patch size means the fastest kernels are found once and reused.
torch.backends.cudnn.benchmark = True

print(f"Models: {MODELS_TO_RUN}")
print(f"Folds : {FOLDS_TO_RUN}")
print(f"Epochs: {EPOCHS} | Batch size: {BS} | LR: {LR} | TTA: {USE_TTA}")
print(f"Target spacing: {TARGET_SPACING} | Patch size: {PATCH_SIZE}")

## 3. Dataset and folds

The dataset CSV lists one row per case, with the raw image and mask paths, the pre-assigned
`fold` (1..5), and a `split` column. Cross-validation defines the validation set for each
fold inside `train_one_fold` as `is_val = (fold == fold_num)`; the held-out fold is the
validation set and the other four folds are training. All 346 cases take part in every
sweep, so the `split` column is deliberately not used here.

In [ ]:
train_df = pd.read_csv(DATA_CSV)
print(f"Total cases: {len(train_df)}")

print("\nCases per fold (each fold is the validation set when held out):")
print(train_df["fold"].value_counts().sort_index().to_string())

print("\n'split' column (present in the CSV but IGNORED by cross-validation):")
print(train_df["split"].value_counts().to_string())

train_df[["case_id", "t1_img_path", "t1_seg_path", "fold", "split"]].head()

## 4. Preprocess once to disk

Preprocessing (RAS+ reorientation, resampling to `TARGET_SPACING`, and foreground-masked
Z-normalization) is **fold-independent**: it depends only on each raw volume, not on which
fold that volume lands in. The original per-model scripts repeated it inside every fold; we
hoist it out and run it a single time over all 346 cases. This is faithful to the pipeline
but avoids four redundant passes per model.

`preprocess_dataset` writes the processed volumes under `preprocessed/` and adds the columns
`t1_img_path_preprocessed` and `t1_seg_path_preprocessed` to `train_df` in place. With
`skip_existing=True` it is effectively idempotent: re-running reuses the cache instead of
recomputing, so you can safely point it at an existing `preprocessed/` folder. During
training, `PatchConfig(preprocessed=True)` then skips reorder, resample, and normalization
because the inputs are already prepared.

### Intensity normalization

`ZNormalization(masking_method="foreground")` applies a Z-score: it subtracts a mean and
divides by a standard deviation. These are computed from foreground voxels, meaning voxels
with intensity above zero (read from the image, not the tumor mask), and then applied to the
whole volume. Restricting the statistics to the foreground stops the large zero-valued
background from dominating them and washing out tissue contrast.

In [ ]:
# Single source of truth for pre-patch / pre-inference intensity normalization. The same
# list is reused when building the training DataLoaders and when running sliding-window
# inference, guaranteeing identical preprocessing on both sides.
pre_patch_tfms = [ZNormalization(masking_method="foreground")]

preprocess_dataset(
    train_df,
    img_col="t1_img_path",
    mask_col="t1_seg_path",
    output_dir="preprocessed",
    target_spacing=TARGET_SPACING,
    transforms=pre_patch_tfms,
    max_workers=32,
)

print("Added columns:", [c for c in train_df.columns if c.endswith("_preprocessed")])
train_df[["t1_img_path", "t1_img_path_preprocessed"]].head()

## 5. Shared pipeline building blocks

Every model is trained through the same two helpers, so differences in results come from the
network, not the data pipeline.

`create_patch_config()` returns a `PatchConfig`, the single source of truth for how patches
are sampled and stitched back together:

- The volumes are large and the tumor is tiny, so we train on **192 x 192 x 48** patches
  rather than whole scans.
- A **label sampler** with `label_probabilities={0: 0.2, 1: 0.8}` draws 80% of patches
  centered on tumor voxels. Without this bias the network would rarely see foreground.
- `samples_per_volume=4`: four patches are drawn each time a volume is visited.
- At evaluation time patches overlap by 50% and are blended with a **Hann window**
  (`aggregation_mode="hann"`) for seamless reconstruction.
- `preprocessed=True` tells the loader the volumes on disk are already reoriented, resampled,
  and normalized, so it does not repeat that work.
- `normalization=pre_patch_tfms` records the intensity normalization (foreground
  Z-normalization) on the config as the single source of truth. The volumes on disk are
  already normalized, so training does not re-apply it under `preprocessed=True`; keeping it
  on the config documents the choice, logs it to the MLflow run, and lets inference rebuild the
  identical transform.
- `keep_largest_component=True` post-processes predictions to the single largest connected
  region, which suits a solitary VS tumor.

In [ ]:
def create_patch_config():
    """Patch sampling and aggregation settings shared by all three models."""
    return PatchConfig(
        patch_size=PATCH_SIZE,
        samples_per_volume=4,
        sampler_type="label",
        label_probabilities={0: 0.2, 1: 0.8},
        patch_overlap=0.5,
        keep_largest_component=True,
        target_spacing=TARGET_SPACING,
        preprocessed=True,
        normalization=pre_patch_tfms,
        aggregation_mode="hann",
        queue_num_workers=16,
        queue_length=1200,
    )


patch_config = create_patch_config()
patch_config

### GPU augmentation

`create_gpu_augmentation()` returns a `GpuPatchAugmentation` that applies spatial and
intensity augmentation to each batch directly on the GPU, which keeps the input pipeline
from becoming the bottleneck. The transforms and probabilities follow nnU-Net conventions:
moderate random affine, anisotropy, flips, gamma, intensity scaling, noise, and blur.
Affine translations are specified in voxels, so the requested millimeter shifts are divided
by the target spacing. Augmentation is applied to the training set only.

In [ ]:
def create_gpu_augmentation():
    """nnU-Net-inspired GPU-batched augmentation applied to training patches."""
    ts = TARGET_SPACING
    return GpuPatchAugmentation(
        affine={
            "scales": (0.7, 1.4),
            "degrees": (5, 5, 30),
            "translation": (25 / ts[0], 25 / ts[1], 5 / ts[2]),
            "default_pad_value": 0.0,
            "p": 0.2,
        },
        anisotropy={"axes": (0, 1, 2), "downsampling": (2, 4), "p": 0.25},
        flip={"axes": (0, 1, 2), "p": 0.5},
        gamma={"log_gamma": (-0.3, 0.3), "p": 0.3},
        intensity_scale={"scale_range": (0.75, 1.25), "p": 0.1},
        noise={"std": 0.1, "p": 0.1},
        blur={"std": (0.5, 1.0), "p": 0.2},
    )

## 6. Model definitions

The three models are registered in a single `MODELS` dictionary. Each entry provides a
`make_model` factory, a `make_loss` factory, and the exact experiment name, checkpoint
filename, and results directory that the sibling inference notebook expects:

| key | experiment | checkpoint | results dir |
|-----|------------|-----------|-------------|
| `unet` | `vs5f_unet` | `best_unet` | `cv_results_unet` |
| `dynunet` | `vs5f_dynunet` | `best_dynunet` | `cv_results_dynunet` |
| `segmamba` | `vs5f_segmamba` | `best_segmamba` | `cv_results_segmamba` |

UNet and SegMamba use a Dice + cross-entropy loss. DynUNet emits predictions at several
decoder depths, so it is wrapped in `DynUNetDSAdapter` (which turns the stacked output into
the list that `DeepSupervisionLoss` expects) and trained with a deep-supervision loss. UNet
and DynUNet are compiled with `torch.compile`; SegMamba is not, because it relies on custom
CUDA kernels.

### Install SegMamba

**UNet and DynUNet need nothing extra** beyond fastMONAI: they are MONAI built-ins.

**SegMamba requires our fork** (`models_segmamba`, imported as `from
models_segmamba.segmambav2 import SegMamba`; default backend `mamba_ssm`, 138.77M
parameters). Install it from GitHub with the `[gpu]` extra, which is the one command you
run for this notebook: it brings the base package plus the CUDA Mamba kernels that the
default `mamba_ssm` backend needs:

    pip install "segmamba-v2[gpu] @ git+https://github.com/skaliy/SegMamba-V2.git"

The kernels (`mamba-ssm`, `causal-conv1d`) sit behind the `[gpu]` extra rather than in the
base install because they compile CUDA extensions and only build where the CUDA toolchain is
present; keeping them optional lets the package install on machines without a GPU. For a
CPU-only box (inference/dev; CPU training is not practical) use `[cpu]` instead, which pulls
in transformers for the `mamba_backend="mambamixer"` path:

    pip install "segmamba-v2[cpu] @ git+https://github.com/skaliy/SegMamba-V2.git"

Caveat: `mamba-ssm` and `causal-conv1d` are CUDA-only and are compiled against the active
torch build. A torch version bump can break `causal-conv1d` with an "undefined symbol" ABI
error; reinstall it against the current torch with `--no-build-isolation --force-reinstall`.

SegMamba is imported once, up front, in the setup cell (section 1) inside a `try/except`. If
the fork is not importable, `SEGMAMBA_AVAILABLE` is set to `False`, the install command is
printed, and the next cell skips SegMamba while UNet and DynUNet still run. A fork that imports
but whose CUDA kernels are broken (the "undefined symbol" case above) is not caught here,
because the import does not load the kernels; it surfaces later, when the model is built, where
the driver's per-(model, fold) `try/except` skips just that run.

In [ ]:
MODELS = {}

def make_unet():
    model = UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        channels=(64, 128, 256, 512, 1024),
        strides=(2, 2, 2, 2),
        num_res_units=4,
        norm=Norm.INSTANCE,
        act=("LEAKYRELU", {"negative_slope": 0.01, "inplace": True}),
    )
    return torch.compile(model)


def make_unet_loss():
    return CustomLoss(loss_func=DiceCELoss(
        to_onehot_y=True, softmax=True, include_background=False, batch=True
    ))


MODELS["unet"] = dict(
    make_model=make_unet, make_loss=make_unet_loss,
    experiment="vs5f_unet", best_fname="best_unet", results_dir="cv_results_unet",
)

def make_dynunet():
    dynunet = DynUNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        kernel_size=[[3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3]],
        strides=[[1, 1, 1], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]],
        upsample_kernel_size=[[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]],
        filters=[64, 128, 256, 512, 1024],
        res_block=True,
        deep_supervision=True,
        deep_supr_num=3,
    )
    model = DynUNetDSAdapter(dynunet)
    return torch.compile(model)


def make_dynunet_loss():
    base_loss = DiceCELoss(
        to_onehot_y=True, softmax=True, include_background=False, batch=True
    )
    return CustomLoss(loss_func=DeepSupervisionLoss(base_loss, weight_mode="exp"))


MODELS["dynunet"] = dict(
    make_model=make_dynunet, make_loss=make_dynunet_loss,
    experiment="vs5f_dynunet", best_fname="best_dynunet", results_dir="cv_results_dynunet",
)

# SegMamba (optional fork): registered only if it imported cleanly in the setup cell. Not
# wrapped in torch.compile because it relies on custom CUDA kernels.
if SEGMAMBA_AVAILABLE:

    def make_segmamba():
        return SegMamba(
            in_chans=1,
            out_chans=2,
            depths=[2, 2, 2, 2],
            feat_size=[48, 96, 192, 384],
            hidden_size=768,
        )

    def make_segmamba_loss():
        return CustomLoss(loss_func=DiceCELoss(
            to_onehot_y=True, softmax=True, include_background=False, batch=True
        ))

    MODELS["segmamba"] = dict(
        make_model=make_segmamba, make_loss=make_segmamba_loss,
        experiment="vs5f_segmamba", best_fname="best_segmamba",
        results_dir="cv_results_segmamba",
    )

print("Registered models:", list(MODELS))

# If segmamba was requested but is unavailable, note it (the setup cell already printed the
# install/fix command). A UNet/DynUNet-only run stays quiet.
if "segmamba" in MODELS_TO_RUN and "segmamba" not in MODELS:
    print("\n[segmamba] requested but its fork is not importable, so it is skipped "
          "(see the install hint from the setup cell above).")

## 7. Train one fold

`train_one_fold(model_key, fold_num, train_df, patch_config)` runs a single training job. It
marks the held-out fold as validation (`is_val = fold == fold_num`), builds the patch
DataLoaders from the **preprocessed** columns, constructs the model and loss from the
registry, and trains with `fit_one_cycle`. Training uses:

- `AccumulatedDice(n_classes=2)` as the monitored metric (nnU-Net-style pseudo-Dice
  accumulated over the whole validation set, more stable than per-batch Dice).
- `EMACheckpoint` to save the best weights by the exponential moving average of that metric,
  under the per-model checkpoint name so models do not overwrite each other.
- `create_mlflow_callback` to log parameters, metrics, the split, and artifacts to the
  per-model MLflow experiment, tagged with the dataset fingerprint.
- `.to_bf16()` for bfloat16 mixed-precision training.

At the end it calls `evaluate_fold` (defined in the next section) and frees GPU memory. The
function is defined here but only executed later by the driver loop, so it can safely refer
to `evaluate_fold` from the following cell.

In [ ]:
def train_one_fold(model_key, fold_num, train_df, patch_config):
    """Train one model on one fold, then evaluate the held-out fold."""
    entry = MODELS[model_key]
    print(f"\n{'='*60}")
    print(f"  {model_key.upper()}  |  FOLD {fold_num}")
    print(f"{'='*60}\n")

    # Fold membership: the held-out fold becomes the validation set. All 346 cases
    # participate; the CSV 'split' column is not used here.
    train_df = train_df.copy()
    train_df["is_val"] = train_df["fold"] == fold_num

    # Content fingerprint of the label set, logged to MLflow for reproducibility.
    med_dataset = MedDataset(
        img_list=train_df.t1_seg_path.tolist(), dtype=MedMask, max_workers=32
    )

    gpu_aug = create_gpu_augmentation()

    # Normalization lives on patch_config (single source of truth). The preprocessed columns
    # are already reoriented, resampled, and Z-normalized, so preprocessed=True skips re-applying
    # it here; the config still carries the spec for logging and inference parity.
    dls = MedPatchDataLoaders.from_df(
        df=train_df,
        img_col="t1_img_path_preprocessed",
        mask_col="t1_seg_path_preprocessed",
        valid_col="is_val",
        patch_config=patch_config,
        gpu_augmentation=gpu_aug,
        bs=BS,
    )

    print(f"[{model_key} fold {fold_num}] Train: {len(dls.train.subjects_dataset)}, "
          f"Val: {len(dls.valid.subjects_dataset)}")

    model = entry["make_model"]()
    loss_func = entry["make_loss"]()

    learn = Learner(dls, model, loss_func=loss_func,
                    metrics=[AccumulatedDice(n_classes=2)]).to_bf16()

    save_best = EMACheckpoint(
        monitor="accumulated_dice", momentum=0.9,
        comp=np.greater, fname=entry["best_fname"], with_opt=False,
    )

    mlflow_cb = create_mlflow_callback(
        learn,
        experiment_name=entry["experiment"],
        run_name=f"fold_{fold_num}",
        extra_tags={"fold": str(fold_num)},
        dataset_version=med_dataset.fingerprint,
    )

    learn.fit_one_cycle(EPOCHS, LR, cbs=[mlflow_cb, save_best])

    results_df = evaluate_fold(
        learn, patch_config, dls, pre_patch_tfms, fold_num,
        tta=USE_TTA, mlflow_cb=mlflow_cb, results_dir=Path(entry["results_dir"]),
    )

    # Release GPU memory before the next (model, fold).
    del learn, model, dls, gpu_aug
    torch.cuda.empty_cache()
    gc.collect()

    return results_df

## 8. Evaluate a fold

After training, we score the held-out fold on the **raw** validation images (not the cached
preprocessed copies), re-applying the identical `ZNormalization` through
`pre_inference_tfms`. This mirrors real inference on unseen data end to end.

For each case we compute Dice, sensitivity, precision, lesion-detection rate, and signed
relative volume error, plus spacing-aware surface metrics (ASSD, HD95, and Normalized Surface
Dice at 0.5/1.0/2.0 mm) via `calculate_surface_metrics`. Voxel spacing is read **per case**
from the ground-truth file, because the cohort spacing is not uniform and a single latched
affine would corrupt every distance. Per-case scores go to `results.csv`, provenance to
`benchmark_meta.json`, and finite-mean aggregates to MLflow.

In [ ]:
def evaluate_fold(learn, patch_config, dls, pre_patch_tfms, fold_num, tta, mlflow_cb, results_dir):
    results_dir = Path(results_dir)
    fold_dir = results_dir / f"fold_{fold_num}"
    pred_dir = fold_dir / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)

    learn.cuda()

    val_df = dls._valid_source_df
    val_img_paths = val_df["t1_img_path"].tolist()
    val_mask_paths = val_df["t1_seg_path"].tolist()

    print(f"[Fold {fold_num}] Running inference on {len(val_img_paths)} validation images...")
    predictions = patch_inference(
        learner=learn,
        config=patch_config,
        file_paths=val_img_paths,
        pre_inference_tfms=pre_patch_tfms,
        save_dir=str(pred_dir),
        progress=True,
        tta=tta,
    )

    # Per-case metric table via the library helper: it reads each GT mask's data AND spacing
    # from one object (so the array and its spacing can't disagree) and runs the full panel.
    results_df = evaluate_segmentations(predictions, val_mask_paths,
                                        case_ids=val_df["case_id"].tolist())
    results_df.insert(1, "image", [Path(p).name for p in val_img_paths])
    if results_df["spacing_mm"].astype(str).nunique() == 1:
        print(f"[Fold {fold_num}] WARNING: spacing_mm is constant across the fold "
              f"({results_df['spacing_mm'].iloc[0]}); verify per-case derivation if unexpected.")
    results_df.to_csv(fold_dir / "results.csv", index=False)

    # Benchmark provenance (arxiv:2410.02630 transparency): metric implementation,
    # NSD tolerances, and per-case spacing/empty-status used for the surface metrics.
    from fastMONAI.vision_metrics import _SURFACE_DISTANCE_SOURCE
    with open(fold_dir / "benchmark_meta.json", "w") as f:
        json.dump({
            "surface_distance_source": _SURFACE_DISTANCE_SOURCE,
            "nsd_tolerances_mm": [0.5, 1.0, 2.0],   # evaluate_segmentations defaults
            "nsd_headline_tau_mm": 1.0,
            "status_counts": results_df["surface_status"].value_counts().to_dict(),
            "per_case": [{"case_id": r["case_id"], "spacing_mm": list(r["spacing_mm"]),
                          "surface_status": r["surface_status"]}
                         for r in results_df.to_dict("records")],
        }, f, indent=2)

    # inf-safe means: one_empty cases score assd_mm/hd95_mm = inf, which would poison a plain
    # mean (and MLflow rejects non-finite values). Replacing inf with NaN lets pandas skip it.
    numeric = results_df.select_dtypes(include="number").replace([np.inf, -np.inf], np.nan)
    mlflow_cb.log_metrics_table(results_df, display=False)
    mlflow_cb.log_metrics({f"val_{m}": numeric[m].mean() for m in numeric.columns})
    mlflow_cb.log_dataframe(results_df)

    print(f"[Fold {fold_num}] Results saved to {fold_dir / 'results.csv'}")
    print(f"[Fold {fold_num}] DSC: {results_df['dsc'].mean():.4f} +/- {results_df['dsc'].std():.4f}")
    return results_df

## 9. Cross-validation driver

The driver sweeps `MODELS_TO_RUN x FOLDS_TO_RUN`, calling `train_one_fold` for each pair.
Each pair is wrapped in try/except: if one job fails (for example an out-of-memory error on
one fold) the error is printed and the sweep continues with the next pair rather than
aborting the whole run. Unregistered model keys (for example `segmamba` without the fork)
are skipped with a message.

This is the long-running cell. For the full paper configuration it trains up to 15 models
and can take days. Reduce the knobs in the configuration cell for a quick demo.

In [ ]:
# Resilient sweep: a failure on one (model, fold) is logged and skipped so it does not abort
# the rest of the run. Re-running is safe; a completed fold simply overwrites its outputs.
for model_key in MODELS_TO_RUN:
    if model_key not in MODELS:
        print(f"[skip] '{model_key}' is not registered (see the model definitions cell above).")
        continue

    Path(MODELS[model_key]["results_dir"]).mkdir(parents=True, exist_ok=True)

    for fold_num in FOLDS_TO_RUN:
        try:
            train_one_fold(model_key, fold_num, train_df, patch_config)
        except Exception as exc:
            print(f"[FAILED] {model_key} fold {fold_num}: {type(exc).__name__}: {exc}")
            traceback.print_exc()
            torch.cuda.empty_cache()
            gc.collect()
            continue

print("\nSweep complete.")

## 10. Aggregate per-model results

For each model, `aggregate_results` concatenates the per-fold `results.csv` files into a
single `cv_summary.csv` inside that model's results directory, then prints a per-metric
mean/standard-deviation summary and the per-fold DSC breakdown. Surface distances in
millimeters (ASSD, HD95) are averaged over **finite values only**: a case where exactly one
of prediction or ground truth is empty scores `+inf` by design, and a plain mean would be
dominated by it.

In [ ]:
def aggregate_results(results_dir):
    """Concatenate a model's per-fold results.csv into cv_summary.csv and print a summary.

    Returns the combined DataFrame, or None if no fold results exist yet (notebook-safe:
    it does not exit the process).
    """
    results_dir = Path(results_dir)
    all_results = []
    for fold_dir in sorted(results_dir.glob("fold_*")):
        csv_path = fold_dir / "results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df["fold"] = int(fold_dir.name.split("_")[1])
            all_results.append(df)

    if not all_results:
        print(f"No fold results found in {results_dir}. Run training first.")
        return None

    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(results_dir / "cv_summary.csv", index=False)

    metrics = ["dsc", "sensitivity", "precision", "ldr", "rve",
               "assd_mm", "hd95_mm", "nsd_tau0.5_mm", "nsd_tau1.0_mm", "nsd_tau2.0_mm"]
    inf_excluded = {"assd_mm", "hd95_mm"}  # one_empty -> inf must not poison the mean
    print(f"\n{'='*60}")
    print(f"  CROSS-VALIDATION SUMMARY: {results_dir.name}")
    print(f"{'='*60}")
    print(f"  Folds completed: {sorted(combined['fold'].unique())}")
    print(f"  Total subjects:  {len(combined)}\n")
    print(f"  {'Metric':<15} {'Mean':>10} {'Std':>10}")
    print(f"  {'-'*35}")
    for m in metrics:
        if m not in combined.columns:
            continue
        col = combined[m]
        note = ""
        if m in inf_excluded:
            mask = np.isfinite(col)
            n_skip = int((~mask).sum())
            col = col[mask]
            if n_skip:
                note = f"  ({n_skip} non-finite excl.)"
        print(f"  {m:<15} {col.mean():>10.4f} {col.std():>10.4f}{note}")

    if "surface_status" in combined.columns:
        print(f"\n  Surface-metric status counts:")
        for status, n in combined["surface_status"].value_counts().items():
            print(f"    {status:<12} {n}")

    print(f"\n  Per-fold DSC:")
    for fold_num, group in combined.groupby("fold"):
        print(f"    Fold {fold_num}: {group['dsc'].mean():.4f} +/- {group['dsc'].std():.4f}")

    print(f"\n  Results saved to {results_dir / 'cv_summary.csv'}")
    return combined

In [ ]:
# Aggregate each model that ran; collect the combined per-case tables for the comparison.
cv_combined = {}
for model_key in MODELS_TO_RUN:
    entry = MODELS.get(model_key)
    if entry is None:
        print(f"[skip] '{model_key}' is not registered.")
        continue
    combined = aggregate_results(Path(entry["results_dir"]))
    if combined is not None:
        cv_combined[model_key] = combined

## 11. Cross-model comparison

This is the headline table and it is net-new relative to the per-model scripts. For every
model that produced results, we pool all held-out cases across folds and report the mean and
standard deviation of each key metric, save the numbers to `cv_model_comparison.csv`, and
display a compact `mean +/- std` view. The same inf-safe averaging is applied here, so the
millimeter surface metrics ignore non-finite per-case scores (and NaN, e.g. RVE on an empty
ground truth, is skipped the same way).

In [ ]:
# Headline metrics for the cross-model table. Surface distances in mm (assd/hd95) can be
# +inf for 'one_empty' cases (exactly one of prediction/ground truth empty). pandas .mean()/
# .std() already skip NaN but not inf, so we replace inf with NaN first; the finite-only mean
# and sample std (ddof=1, the pandas default) then match the per-model summary above.
KEY_METRICS = ["dsc", "sensitivity", "precision", "ldr", "rve",
               "assd_mm", "hd95_mm", "nsd_tau1.0_mm"]

rows = []
for model_key, combined in cv_combined.items():
    row = {"model": model_key,
           "folds": sorted(combined["fold"].unique().tolist()),
           "n_cases": int(len(combined))}
    for m in KEY_METRICS:
        if m in combined.columns:
            finite = combined[m].replace([np.inf, -np.inf], np.nan)  # drop inf; pandas skips NaN
            row[f"{m}_mean"] = float(finite.mean())
            row[f"{m}_std"] = float(finite.std())
    rows.append(row)

if rows:
    comparison_df = pd.DataFrame(rows).set_index("model")
    comparison_df.to_csv("cv_model_comparison.csv")
    print("Saved cross-model comparison to cv_model_comparison.csv\n")

    # Compact 'mean +/- std' view of the headline metrics.
    pretty = pd.DataFrame(index=comparison_df.index)
    for m in KEY_METRICS:
        mc, sc = f"{m}_mean", f"{m}_std"
        if mc in comparison_df.columns:
            pretty[m] = [f"{mu:.4f} +/- {sd:.4f}"
                         for mu, sd in zip(comparison_df[mc], comparison_df[sc])]
    display(pretty)
else:
    print("No per-model results available yet. Run the driver loop first.")

## 12. Viewing runs and next steps

Every fold logs to MLflow (metrics, parameters, the train/validation split, and model
artifacts) under the per-model experiment names `vs5f_unet`, `vs5f_dynunet`, and
`vs5f_segmamba`. To browse them, launch the fastMONAI MLflow UI from a notebook cell:

    mlflow_ui = MLflowUIManager()
    mlflow_ui.start_ui()   # opens http://localhost:5001

The UI is tied to the kernel and is reaped when the interpreter exits, so restarting the
notebook leaves no orphaned server holding the port.

Artifacts written to disk:

- `preprocessed/` - reoriented, resampled, normalized volumes (shared by all folds).
- `cv_results_<model>/fold_N/` - per-fold `results.csv`, `benchmark_meta.json`, and NIfTI
  predictions.
- `cv_results_<model>/cv_summary.csv` - all folds concatenated for one model.
- `cv_model_comparison.csv` - the headline mean +/- std table across models.

The best checkpoint for each model is saved by `EMACheckpoint` as `best_unet`,
`best_dynunet`, and `best_segmamba`. To run any trained model on brand-new cases, see the
sibling notebook `02_inference_new_cases.ipynb`, which reuses the same `TARGET_SPACING`,
`PATCH_SIZE`, and `ZNormalization(masking_method="foreground")` contract so that inference
preprocessing matches training exactly.

To deploy the five folds together as a soft-vote ensemble, pass the list of fold learners to `patch_inference` (or `PatchInferenceEngine`): it averages their per-patch probabilities in a single sliding-window pass, which usually beats any single fold. Notebook 02 does exactly this.

## References

- Connor, S., et al. (2025). The Real-World Impact of Vestibular Schwannoma Fully Automated Volume Measures on the Evaluation of Size Change and Clinical Management Outcomes in a Multidisciplinary Meeting Setting. *Journal of International Advanced Otology*. https://doi.org/10.5152/iao.2025.241693
- Dhayalan, D., et al. (2023). Upfront Radiosurgery vs a Wait-and-Scan Approach for Small- or Medium-Sized Vestibular Schwannoma: The V-REX Randomized Clinical Trial. *JAMA*. https://doi.org/10.1001/jama.2023.12222
- Dorent, R., et al. (2023). CrossMoDA 2021 challenge: Benchmark of cross-modality domain adaptation techniques for vestibular schwannoma and cochlea segmentation. *Medical Image Analysis*. https://doi.org/10.1016/j.media.2022.102628
- Häußler, S. M., et al. (2025). Automatic Segmentation of Vestibular Schwannoma From MRI Using Two Cascaded Deep Learning Networks. *The Laryngoscope*. https://doi.org/10.1002/lary.31979
- Isensee, F., et al. (2024). nnU-Net Revisited: A Call for Rigorous Validation in 3D Medical Image Segmentation. *MICCAI 2024*. https://doi.org/10.1007/978-3-031-72114-4_47
- Kujawa, A., et al. (2024). Deep learning for automatic segmentation of vestibular schwannoma: a retrospective study from multi-center routine MRI. *Frontiers in Computational Neuroscience*. https://doi.org/10.3389/fncom.2024.1365727
- Shapey, J., et al. (2021). Segmentation of vestibular schwannoma from MRI, an open annotated dataset and baseline algorithm. *Scientific Data*. https://doi.org/10.1038/s41597-021-01064-w